In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


### Import libraries

In [2]:
from pathlib import Path
import pandas as pd

### Set project paths

In [7]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() in ("notebook", "notebooks"):
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"Dataset folder not found: {RAW_DIR}"


### Define image categories

In [4]:
EXPECTED_CLASSES = [
    "bacterial_leaf_blight",
    "brown_spot",
    "Healthy",
    "leaf_blast",
    "leaf_scald",
    "narrow_brown_spot",
    "neck_blast",
    "rice_hispa",
    "sheath_blight",
    "tungro"
]

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}


### Create the CSV dataset

In [5]:
records = []

missing_classes = []

for class_name in EXPECTED_CLASSES:

    class_dir = RAW_DIR / class_name

    if not class_dir.exists():
        missing_classes.append(class_name)
        continue

    for image_path in class_dir.rglob("*"):

        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:

            records.append({
                "image_path": image_path.relative_to(ROOT).as_posix(),
                "label": class_name
            })

if missing_classes:
    raise ValueError(f"Missing folders: {missing_classes}")

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No images found. Check your raw dataset folders.")

df = df.sort_values(["label", "image_path"]).reset_index(drop=True)

csv_path = PROCESSED_DIR / "rice_leaf_images.csv"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Total images:", len(df))
print("CSV location:", csv_path)

display(df.head(10))


Total images: 6712
CSV location: d:\My_Project\Analyse\AI-Paddy-Farming-Intelligence\data\processed\rice_leaf_images.csv


,image_path,label
0,data/raw/Healthy/Healthy_rice_leaf (104).jpg,Healthy
1,data/raw/Healthy/Healthy_rice_leaf (105).jpg,Healthy
2,data/raw/Healthy/Healthy_rice_leaf (106).jpg,Healthy
3,data/raw/Healthy/Healthy_rice_leaf (107).jpg,Healthy
4,data/raw/Healthy/Healthy_rice_leaf (108).jpg,Healthy
5,data/raw/Healthy/Healthy_rice_leaf (112).jpg,Healthy
6,data/raw/Healthy/Healthy_rice_leaf (113).jpg,Healthy
7,data/raw/Healthy/Healthy_rice_leaf (114).jpg,Healthy
8,data/raw/Healthy/Healthy_rice_leaf (115).jpg,Healthy
9,data/raw/Healthy/Healthy_rice_leaf (116).jpg,Healthy


### Count images per category

In [6]:
class_counts = df["label"].value_counts().sort_index()

display(class_counts.to_frame("image_count"))

print("\nTotal categories:", df["label"].nunique())
print("Total images:", len(df))

,image_count
label,
Healthy,753
bacterial_leaf_blight,785
brown_spot,761
leaf_blast,521
leaf_scald,514
narrow_brown_spot,767
neck_blast,677
rice_hispa,770
sheath_blight,608



Total categories: 10
Total images: 6712
